In [13]:
import os
import shutil
import subprocess
from pathlib import Path

os.environ["KAGGLE_USERNAME"] = "samanbehzad"
os.environ["KAGGLE_KEY"] = "PASTE_KEY"

subprocess.check_call(["pip", "install", "-q", "-U", "kaggle"])
dest = Path("/kaggle/working/old_output")
if dest.exists():
    shutil.rmtree(dest)
dest.mkdir(parents=True)
pattern = r"pairs\.jsonl|judge_pairwise_kaggle\.py|summarize_judgments\.py"
subprocess.check_call([
    "kaggle", "kernels", "output", "samanbehzad/notebook41f992f9d8",
    "-p", str(dest), "-o", "--file-pattern", pattern, "--page-size", "200",
])
project = Path("/kaggle/working/small-model-alignment")
for path in dest.rglob("*"):
    if not path.is_file():
        continue
    rel = path.as_posix()
    marker = "small-model-alignment/"
    if marker not in rel:
        continue
    target = project / rel.split(marker, 1)[1]
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(path, target)
    print("copied", target.relative_to(project))
for name in [
    "outputs/eval/dpo_ultra_vs_sft_pairs.jsonl",
    "outputs/eval/dpo_hh_vs_sft_pairs.jsonl",
    "outputs/eval/lora_vs_smol_pairs.jsonl",
    "scripts/judge_pairwise_kaggle.py",
    "scripts/summarize_judgments.py",
]:
    print("OK" if (project / name).exists() else "MISSING", name)

Output file downloaded to /kaggle/working/old_output/small-model-alignment/outputs/eval/dpo_hh_vs_sft_pairs.jsonl
Output file downloaded to /kaggle/working/old_output/small-model-alignment/outputs/eval/dpo_ultra_vs_sft_pairs.jsonl
Output file downloaded to /kaggle/working/old_output/small-model-alignment/outputs/eval/lora_vs_smol_pairs.jsonl
Output file downloaded to /kaggle/working/old_output/small-model-alignment/scripts/judge_pairwise_kaggle.py
Output file downloaded to /kaggle/working/old_output/small-model-alignment/scripts/summarize_judgments.py
Kernel log downloaded to /kaggle/working/old_output/notebook41f992f9d8.log 
copied scripts/summarize_judgments.py
copied scripts/judge_pairwise_kaggle.py
copied outputs/eval/dpo_hh_vs_sft_pairs.jsonl
copied outputs/eval/lora_vs_smol_pairs.jsonl
copied outputs/eval/dpo_ultra_vs_sft_pairs.jsonl
OK outputs/eval/dpo_ultra_vs_sft_pairs.jsonl
OK outputs/eval/dpo_hh_vs_sft_pairs.jsonl
OK outputs/eval/lora_vs_smol_pairs.jsonl
OK scripts/judge_pai

In [15]:
%%bash
cd /kaggle/working/small-model-alignment
python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_ultra_vs_sft_pairs.jsonl --output_path outputs/eval/dpo_ultra_vs_sft_judged.jsonl --judge_model google/gemini-3.6-flash --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments --rubric helpfulness
python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_hh_vs_sft_pairs.jsonl --output_path outputs/eval/dpo_hh_vs_sft_judged.jsonl --judge_model google/gemini-3.6-flash --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments --rubric safety
python scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/lora_vs_smol_pairs.jsonl --output_path outputs/eval/lora_vs_smol_judged.jsonl --judge_model google/gemini-3.6-flash --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments --rubric persona
python scripts/summarize_judgments.py --judged_path outputs/eval/dpo_ultra_vs_sft_judged.jsonl --candidate dpo --output_path outputs/eval/dpo_ultra_summary.json
python scripts/summarize_judgments.py --judged_path outputs/eval/dpo_hh_vs_sft_judged.jsonl --candidate dpo --output_path outputs/eval/dpo_hh_summary.json
python scripts/summarize_judgments.py --judged_path outputs/eval/lora_vs_smol_judged.jsonl --candidate lora --output_path outputs/eval/lora_summary.json

judging 0 pairs with google/gemini-3.6-flash; 300 already saved
judged file: outputs/eval/dpo_ultra_vs_sft_judged.jsonl
failure file: outputs/eval/dpo_ultra_vs_sft_judged.failures.jsonl
judging 0 pairs with google/gemini-3.6-flash; 300 already saved
judged file: outputs/eval/dpo_hh_vs_sft_judged.jsonl
failure file: outputs/eval/dpo_hh_vs_sft_judged.failures.jsonl
judging 0 pairs with google/gemini-3.6-flash; 16 already saved
judged file: outputs/eval/lora_vs_smol_judged.jsonl
failure file: outputs/eval/lora_vs_smol_judged.failures.jsonl
{
  "candidate": "dpo",
  "sample_count": 300,
  "wins": 104,
  "losses": 141,
  "ties": 54,
  "unknown": 1,
  "failures": 0,
  "win_rate_including_ties": 0.3466666666666667,
  "win_rate_excluding_ties": 0.42448979591836733,
  "judge_model": "google/gemini-3.6-flash",
  "by_category": {
    "helpful_base": {
      "candidate": "dpo",
      "sample_count": 49,
      "wins": 18,
      "losses": 19,
      "ties": 11,
      "unknown": 1,
      "failures": 0